In [1]:
import sys
import os

# Agrega la carpeta raíz al path de Python
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [4]:
# scripts/ingestion.py

import os
from config.spark_session import get_spark_session

# Configuración inicial
DATA_PATH = os.path.abspath(os.path.join(os.getcwd(), '..', 'data'))

def main():
    # Levantar sesión de Spark
    spark = get_spark_session("IngestionOnboardingFintech")

    # Cargar datasets
    lk_users = spark.read.option("header", "true").csv(os.path.join(DATA_PATH, 'lk_users.csv'))
    bt_users_transactions = spark.read.option("header", "true").csv(os.path.join(DATA_PATH, 'bt_users_transactions.csv'))
    lk_onboarding = spark.read.option("header", "true").csv(os.path.join(DATA_PATH, 'lk_onboarding.csv'))

    # Mostrar algunas estadísticas rápidas
    print("\n=== lk_users ===")
    lk_users.printSchema()
    print(f"Cantidad de registros: {lk_users.count()}")

    print("\n=== bt_users_transactions ===")
    bt_users_transactions.printSchema()
    print(f"Cantidad de registros: {bt_users_transactions.count()}")

    print("\n=== lk_onboarding ===")
    lk_onboarding.printSchema()
    print(f"Cantidad de registros: {lk_onboarding.count()}")

    # Opcional: guardar como parquet para usar más rápido después
    output_path = os.path.join(DATA_PATH, 'processed')
    os.makedirs(output_path, exist_ok=True)

    lk_users.write.mode("overwrite").parquet(os.path.join(output_path, 'lk_users.parquet'))
    bt_users_transactions.write.mode("overwrite").parquet(os.path.join(output_path, 'bt_users_transactions.parquet'))
    lk_onboarding.write.mode("overwrite").parquet(os.path.join(output_path, 'lk_onboarding.parquet'))

    print("\nArchivos guardados en formato Parquet.")

if __name__ == "__main__":
    main()


25/04/28 16:54:21 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.



=== lk_users ===
root
 |-- _c0: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- address: string (nullable = true)
 |-- birth_dt: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- type: string (nullable = true)
 |-- rubro: string (nullable = true)

Cantidad de registros: 39000

=== bt_users_transactions ===
root
 |-- _c0: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- transaction_dt: string (nullable = true)
 |-- type: string (nullable = true)
 |-- segment: string (nullable = true)

Cantidad de registros: 7675

=== lk_onboarding ===
root
 |-- _c0: string (nullable = true)
 |-- Unnamed: 0: string (nullable = true)
 |-- first_login_dt: string (nullable = true)
 |-- week_year: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- habito: string (nullable = true)
 |-- habito_dt: string (nullable = true)
 |-- activacion: string (nullable = true)

25/04/28 16:54:24 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , user_id, name, email, address, birth_dt, phone, type, rubro
 Schema: _c0, user_id, name, email, address, birth_dt, phone, type, rubro
Expected: _c0 but found: 
CSV file: file:///Users/juanignaciomagarinoscastro/Desktop/bigdata_project_itba/data/lk_users.csv
25/04/28 16:54:25 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , user_id, transaction_dt, type, segment
 Schema: _c0, user_id, transaction_dt, type, segment
Expected: _c0 but found: 
CSV file: file:///Users/juanignaciomagarinoscastro/Desktop/bigdata_project_itba/data/bt_users_transactions.csv



Archivos guardados en formato Parquet.


25/04/28 16:54:25 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Unnamed: 0, first_login_dt, week_year, user_id, habito, habito_dt, activacion, activacion_dt, setup, setup_dt, return, return_dt
 Schema: _c0, Unnamed: 0, first_login_dt, week_year, user_id, habito, habito_dt, activacion, activacion_dt, setup, setup_dt, return, return_dt
Expected: _c0 but found: 
CSV file: file:///Users/juanignaciomagarinoscastro/Desktop/bigdata_project_itba/data/lk_onboarding.csv


In [5]:
# scripts/preprocessing.py

import os
from pyspark.sql.functions import col, to_date
from pyspark.sql.types import IntegerType
from config.spark_session import get_spark_session

# Configuración de paths
DATA_PATH = os.path.abspath(os.path.join(os.getcwd(), '..', 'data', 'processed'))

def main():
    # Levantar sesión de Spark
    spark = get_spark_session("PreprocessingOnboardingFintech")

    # Leer los archivos Parquet procesados
    lk_users = spark.read.parquet(os.path.join(DATA_PATH, 'lk_users.parquet'))
    bt_users_transactions = spark.read.parquet(os.path.join(DATA_PATH, 'bt_users_transactions.parquet'))
    lk_onboarding = spark.read.parquet(os.path.join(DATA_PATH, 'lk_onboarding.parquet'))

    # --- LIMPIEZAS ---

    # 1. Eliminar columnas innecesarias
    lk_users = lk_users.drop("_c0")
    bt_users_transactions = bt_users_transactions.drop("_c0")
    lk_onboarding = lk_onboarding.drop("_c0", "Unnamed: 0")

    # 2. Convertir campos de fecha a tipo Date
    bt_users_transactions = bt_users_transactions.withColumn("transaction_dt", to_date("transaction_dt", "yyyy-MM-dd"))
    lk_onboarding = lk_onboarding.withColumn("first_login_dt", to_date("first_login_dt", "yyyy-MM-dd")) \
                                 .withColumn("habito_dt", to_date("habito_dt", "yyyy-MM-dd")) \
                                 .withColumn("activacion_dt", to_date("activacion_dt", "yyyy-MM-dd")) \
                                 .withColumn("setup_dt", to_date("setup_dt", "yyyy-MM-dd")) \
                                 .withColumn("return_dt", to_date("return_dt", "yyyy-MM-dd"))

    # 3. Convertir flags (habito, activacion, setup, return) a Integer
    for col_name in ["habito", "activacion", "setup", "return"]:
        lk_onboarding = lk_onboarding.withColumn(col_name, col(col_name).cast(IntegerType()))

    # --- GUARDAR VERSION LIMPIA ---
    output_clean_path = os.path.join(DATA_PATH, 'clean')
    os.makedirs(output_clean_path, exist_ok=True)

    lk_users.write.mode("overwrite").parquet(os.path.join(output_clean_path, 'lk_users_clean.parquet'))
    bt_users_transactions.write.mode("overwrite").parquet(os.path.join(output_clean_path, 'bt_users_transactions_clean.parquet'))
    lk_onboarding.write.mode("overwrite").parquet(os.path.join(output_clean_path, 'lk_onboarding_clean.parquet'))

    print("\nPreprocesamiento terminado y archivos limpios guardados.")

if __name__ == "__main__":
    main()


25/04/28 16:56:14 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
25/04/28 16:56:15 ERROR Utils: Aborting task
org.apache.spark.SparkUpgradeException: [INCONSISTENT_BEHAVIOR_CROSS_VERSION.PARSE_DATETIME_BY_NEW_PARSER] You may get a different result due to the upgrading to Spark >= 3.0:
Fail to parse '2022-01-20 23:05:07.884739087' in the new parser. You can set "spark.sql.legacy.timeParserPolicy" to "LEGACY" to restore the behavior before Spark 3.0, or set to "CORRECTED" and treat it as an invalid datetime string.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.failToParseDateTimeInNewParserError(QueryExecutionErrors.scala:1368)
	at org.apache.spark.sql.catalyst.util.DateTimeFormatterHelper$$anonfun$checkParsedDiff$1.applyOrElse(DateTimeFormatterHelper.scala:149)
	at org.apache.spark.sql.catalyst.util.DateTimeFormatterHelper$$anonfun$checkParsedDiff$1.applyOrElse(DateTimeFormatterHelper.scala:142)
	at scala.runtime.AbstractPar

Py4JJavaError: An error occurred while calling o133.parquet.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 19.0 failed 1 times, most recent failure: Lost task 0.0 in stage 19.0 (TID 16) (192.168.0.101 executor driver): org.apache.spark.SparkException: [TASK_WRITE_FAILED] Task failed while writing rows to file:/Users/juanignaciomagarinoscastro/Desktop/bigdata_project_itba/data/processed/clean/bt_users_transactions_clean.parquet.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.taskFailedWhileWritingRowsError(QueryExecutionErrors.scala:788)
	at org.apache.spark.sql.execution.datasources.FileFormatWriter$.executeTask(FileFormatWriter.scala:420)
	at org.apache.spark.sql.execution.datasources.WriteFilesExec.$anonfun$doExecuteWrite$1(WriteFiles.scala:100)
	at org.apache.spark.rdd.RDD.$anonfun$mapPartitionsInternal$2(RDD.scala:888)
	at org.apache.spark.rdd.RDD.$anonfun$mapPartitionsInternal$2$adapted(RDD.scala:888)
	at org.apache.spark.rdd.MapPartit